In [234]:
print("hey")

hey


In [235]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [236]:
from langchain_groq import ChatGroq
import os

In [237]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'model_card': 'https://openai.com/index/gpt-oss-model-card', 'id': 'openai/gpt-oss-120b', 'object': 'model', 'created': 1754408224, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536, 'hugging_face_id': 'openai/gpt-oss-120b', 'name': 'GPT OSS 120B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 65536, 'pricing': {'prompt': '0.00000015', 'completion': '0.0000006', 'image': '0', 'request': '0', 'input_cache_read': '0.000000075'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['tools', 'json_mode', 'structured_outputs', 'reasoning']}, {'id': 'qwen/qwen3.8-27b', 'object': 'model', 'created': 1786984846, 'owned_by': 'Alibaba Cloud', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 16384, 'name': 'Qwen/Qwen3.8-27B', 'input_modalities': 

In [238]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [239]:
llm.invoke("What is the llm?")

AIMessage(content='**LLM** stands for **Large Language Model**.  \n\nA large language model is a type of artificial‑intelligence system that:\n\n1. **Learns from massive text data** – It is trained on billions (or even trillions) of words from books, articles, websites, code, etc.\n2. **Uses deep neural networks** – Most LLMs are built on transformer architectures, which excel at handling sequences of words and capturing long‑range relationships.\n3. **Generates or understands language** – After training, the model can:\n   - Complete a sentence or paragraph,\n   - Answer questions,\n   - Summarize text,\n   - Translate between languages,\n   - Write code,\n   - Perform many other language‑related tasks, often with just a short prompt (few‑shot or zero‑shot learning).\n\n### Why “large”?\n- **Scale of data**: Hundreds of gigabytes to petabytes of text.\n- **Scale of parameters**: Modern LLMs have anywhere from hundreds of millions to hundreds of billions of learnable weights. More para

In [240]:
from langchain.tools import tool

@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

In [241]:
search_database.invoke({
    "query": "customer name",
    "limit": 5
})

"Found 5 results for 'customer name'"

In [242]:
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_openai import ChatOpenAI

# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, name: str, age: int, email: str, runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, {"name": name, "age": age, "email": email})
    return "Successfully saved user info."


store = InMemoryStore()
agent = create_agent(
    llm,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})



{'messages': [HumanMessage(content='Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev', additional_kwargs={}, response_metadata={}, id='3cf444de-d424-4bc1-bd4a-0e30baea9a23'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call save_user_info with given data.', 'tool_calls': [{'id': 'fc_9f0a2da8-f00c-45c3-bff2-316040961488', 'function': {'arguments': '{"age":25,"email":"foo@langchain.dev","name":"Foo","user_id":"abc123"}', 'name': 'save_user_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 179, 'total_tokens': 246, 'completion_time': 0.141250535, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.02153118, 'prompt_tokens_details': None, 'queue_time': 0.400055715, 'total_time': 0.162781715}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e1a78f200e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, '

In [243]:
# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

{'messages': [HumanMessage(content="Get user info for user with id 'abc123'", additional_kwargs={}, response_metadata={}, id='e45fb70a-43db-4d2a-b47b-a3463ea5bd5f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to call get_user_info function with user_id 'abc123'.", 'tool_calls': [{'id': 'fc_c3741cca-0be0-47bd-96e7-2c809a2ddccc', 'function': {'arguments': '{"user_id":"abc123"}', 'name': 'get_user_info'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 164, 'total_tokens': 210, 'completion_time': 0.096500865, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.063881235, 'prompt_tokens_details': None, 'queue_time': 0.359284544, 'total_time': 0.1603821}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16887-92eb-7900-bcbc-37d03c1cc463-0', tool_

In [244]:

import requests
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool


class WeatherInput(BaseModel):
    city: str = Field(description="City name")


def get_weather(city: str) -> str:
    """Fetch current weather for a city."""

    # Step 1: Convert city name to latitude and longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={"name": city, "count": 1, "language": "en"},
        timeout=10,
    )
    geo_response.raise_for_status()
    places = geo_response.json().get("results", [])

    if not places:
        return f"City not found: {city}"

    place = places[0]
    latitude = place["latitude"]
    longitude = place["longitude"]

    # Step 2: Fetch current weather
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_response = requests.get(
        weather_url,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": (
                "temperature_2m,"
                "relative_humidity_2m,"
                "apparent_temperature,"
                "wind_speed_10m"
            ),
            "timezone": "auto",
        },
        timeout=10,
    )
    weather_response.raise_for_status()
    current = weather_response.json()["current"]

    return (
        f"Weather in {place['name']}: "
        f"Temperature: {current['temperature_2m']} °C, "
        f"Feels like: {current['apparent_temperature']} °C, "
        f"Humidity: {current['relative_humidity_2m']}%, "
        f"Wind speed: {current['wind_speed_10m']} km/h, "
        f"Observation time: {current['time']}"
    )


weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="get_weather",
    description="Fetch current weather data for a city",
    args_schema=WeatherInput,
)

# Test the tool directly
print(weather_tool.invoke({"city": "Kozhikode"}))


Weather in Kozhikode: Temperature: 25.0 °C, Feels like: 31.2 °C, Humidity: 98%, Wind speed: 2.6 km/h, Observation time: 2026-10-23T17:00


In [245]:

from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[weather_tool],
    system_prompt="You are a helpful weather assistant."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the weather in Kozhikode?"
        }
    ]
})

print(result["messages"][-1].content)


Sure! Here’s the current weather for **Kozhikode**:

- **Temperature:** 25 °C  
- **Feels like:** 31.2 °C  
- **Humidity:** 98 %  
- **Wind speed:** 2.6 km/h  
- **Observation time:** 2026‑10‑23 17:00 (local time)

It’s quite warm and very humid right now, with light breezes. If you’re heading out, you might want to stay hydrated and consider lighter clothing. Let me know if you need a forecast for the next few days or any other details!


In [246]:
import requests

class WeatherForecastTool:
    def __init__(self):
        self.api_key = os.environ.get("OPENWEATHERMAP_API_KEY")
        self.base_url = "https://api.openweathermap.org/data/2.5"
        self.geourl="https://api.openweathermap.org/geo/1.0/direct"

    def get_current_weather(self, place:str):
        """Get current weather using coordinates."""

        geo_url=f"{self.geourl}"

        geo_params = {
            "q": place,
            "appid": self.api_key}
        
        geo_response = requests.get(geo_url, params=geo_params, timeout=10)
        geo_response.raise_for_status()
        geo_data = geo_response.json()[0]  # Get the first result
        base_url = f"{self.base_url}/weather"
       
       
        params = {
            "lat": geo_data["lat"],
            "lon": geo_data["lon"],
            "appid": self.api_key,
            "units": "metric",
        }

        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()

        return response.json()
    
    def get_forecast_weather(self, place:str):
        """Get weather forecast of a place"""
        try:
            url = f"{self.base_url}/forecast"
            params = {
                "q": place,
                "appid": self.api_key,
               
                "units": "metric"
            }
            response = requests.get(url, params=params)
            return response.json() if response.status_code == 200 else {}
        except Exception as e:
            raise e

In [247]:
weather_data = WeatherForecastTool().get_current_weather(place="london")
weather_data

{'coord': {'lon': -0.1278, 'lat': 51.5074},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 11.93,
  'feels_like': 11.63,
  'temp_min': 11.93,
  'temp_max': 11.93,
  'pressure': 991,
  'humidity': 94,
  'sea_level': 991,
  'grnd_level': 987},
 'visibility': 2175,
 'wind': {'speed': 4.81, 'deg': 316, 'gust': 10.08},
 'clouds': {'all': 100},
 'dt': 1792754624,
 'sys': {'country': 'GB', 'sunrise': 1792737462, 'sunset': 1792774303},
 'timezone': 3600,
 'id': 2643743,
 'name': 'London',
 'cod': 200}

In [248]:
forcast_data = WeatherForecastTool().get_forecast_weather(place="london")
forcast_data

{'cod': '200',
 'message': 0,
 'cnt': 40,
 'list': [{'dt': 1792756800,
   'main': {'temp': 12,
    'feels_like': 11.71,
    'temp_min': 12,
    'temp_max': 12,
    'pressure': 991,
    'sea_level': 991,
    'grnd_level': 989,
    'humidity': 94,
    'temp_kf': 0,
    'dew_point': 11.07},
   'weather': [{'id': 501,
     'main': 'Rain',
     'description': 'moderate rain',
     'icon': '10d'}],
   'clouds': {'all': 100},
   'wind': {'speed': 5, 'deg': 297, 'gust': 11.41},
   'visibility': 3378,
   'pop': 1,
   'rain': {'3h': 8.5},
   'sys': {'pod': 'd'},
   'dt_txt': '2026-10-23 12:00:00'},
  {'dt': 1792767600,
   'main': {'temp': 11.46,
    'feels_like': 11.01,
    'temp_min': 10.37,
    'temp_max': 11.46,
    'pressure': 993,
    'sea_level': 993,
    'grnd_level': 994,
    'humidity': 90,
    'temp_kf': 1.09,
    'dew_point': 9.88},
   'weather': [{'id': 500,
     'main': 'Rain',
     'description': 'light rain',
     'icon': '10d'}],
   'clouds': {'all': 100},
   'wind': {'speed': 7.

In [249]:

if forcast_data:
    forecasts = forcast_data.get("list", [])
    forecast_str = f"Weather forecast in london for the next  days:\n"
    for forecast in forecasts:
        date = forecast.get('dt_txt', 'N/A')
        temp = forecast.get('main', {}).get('temp', 'N/A')

        desc = forecast.get('weather', [{}])[0].get('description', 'N/A')
        forecast_str += f"{date}: {temp}°C, {desc}\n"
        
    print(forecast_str)

        
        

Weather forecast in london for the next  days:
2026-10-23 12:00:00: 12°C, moderate rain
2026-10-23 15:00:00: 11.46°C, light rain
2026-10-23 18:00:00: 10.48°C, overcast clouds
2026-10-23 21:00:00: 9.38°C, overcast clouds
2026-10-24 00:00:00: 8.57°C, clear sky
2026-10-24 03:00:00: 7.53°C, overcast clouds
2026-10-24 06:00:00: 7.57°C, overcast clouds
2026-10-24 09:00:00: 9.86°C, overcast clouds
2026-10-24 12:00:00: 11.91°C, light rain
2026-10-24 15:00:00: 12.17°C, moderate rain
2026-10-24 18:00:00: 13.76°C, moderate rain
2026-10-24 21:00:00: 14.76°C, light rain
2026-10-25 00:00:00: 11.49°C, light rain
2026-10-25 03:00:00: 10.71°C, clear sky
2026-10-25 06:00:00: 9.85°C, clear sky
2026-10-25 09:00:00: 10.89°C, clear sky
2026-10-25 12:00:00: 12.69°C, clear sky
2026-10-25 15:00:00: 12.64°C, clear sky
2026-10-25 18:00:00: 10.53°C, clear sky
2026-10-25 21:00:00: 9.74°C, clear sky
2026-10-26 00:00:00: 8.33°C, overcast clouds
2026-10-26 03:00:00: 8.71°C, overcast clouds
2026-10-26 06:00:00: 8.35°C

In [250]:
from collections import defaultdict

In [251]:
def format_forecast_data(forecast_data):
    """Format the forecast data into a more readable structure."""
    daily = defaultdict(list)
    
    for forecast in forecast_data.get("list", []):
        date = forecast.get('dt_txt', 'N/A').split(' ')[0]  # Get only the date part
        daily[date].append(forecast)

    result = {
        "data": []
    }

    for date, forecasts in daily.items():
        temps = [f.get('main', {}).get('temp', 'N/A') for f in forecasts]
        descriptions = [f.get('weather', [{}])[0].get('description', 'N/A') for f in forecasts]
        description=max(set(descriptions), key=descriptions.count)  # Most common description

        result["data"].append({
            "date": date,
            "min_temp":f'{min(temps) if temps else None}°C',
            "max_temp": f'{max(temps) if temps else None}°C',
            "description": description
        })

    return json.dumps(result, indent=2, ensure_ascii=False)

In [252]:
import json
formated_forecast = format_forecast_data(forcast_data)
print(formated_forecast)

{
  "data": [
    {
      "date": "2026-10-23",
      "min_temp": "9.38°C",
      "max_temp": "12°C",
      "description": "overcast clouds"
    },
    {
      "date": "2026-10-24",
      "min_temp": "7.53°C",
      "max_temp": "14.76°C",
      "description": "overcast clouds"
    },
    {
      "date": "2026-10-25",
      "min_temp": "9.74°C",
      "max_temp": "12.69°C",
      "description": "clear sky"
    },
    {
      "date": "2026-10-26",
      "min_temp": "8.33°C",
      "max_temp": "15.37°C",
      "description": "overcast clouds"
    },
    {
      "date": "2026-10-27",
      "min_temp": "10.26°C",
      "max_temp": "15.88°C",
      "description": "light rain"
    },
    {
      "date": "2026-10-28",
      "min_temp": "8.59°C",
      "max_temp": "11.01°C",
      "description": "clear sky"
    }
  ]
}


In [254]:
formated_forecast = json.loads(formated_forecast)
if formated_forecast:
    forecast_str = "Weather forecast in London for the next 5 days:\n"

    for data in formated_forecast["data"]:
        date = data["date"]
        min_temp = data["min_temp"]
        max_temp = data["max_temp"]
        desc = data["description"]

        forecast_str += (
            f"{date}: Min {min_temp}, Max {max_temp}, "
            f"{desc}\n"
        )

    print(forecast_str)


Weather forecast in London for the next 5 days:
2026-10-23: Min 9.38°C, Max 12°C, overcast clouds
2026-10-24: Min 7.53°C, Max 14.76°C, overcast clouds
2026-10-25: Min 9.74°C, Max 12.69°C, clear sky
2026-10-26: Min 8.33°C, Max 15.37°C, overcast clouds
2026-10-27: Min 10.26°C, Max 15.88°C, light rain
2026-10-28: Min 8.59°C, Max 11.01°C, clear sky

